<a href="https://colab.research.google.com/github/harmanjotk549-bot/Resume-Fitment-Scoring/blob/main/Chitkara_MBA_HR_Resume_Fitment_Scoring_using_TF_IDF%2C_Cosine_Similarity%2C_and_Weighted_Scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Case Study Objective

A company wants to hire a Data Analyst.
There are multiple candidates, and HR wants to identify which candidate is the best fit for the role using a simple AI-based scoring system.

This lab will help students understand:

how resumes can be matched with a job description
how text similarity works
how feature-based scoring works
how final ranking is generated

## Title: Business Problem

A company is hiring for the role of Data Analyst.
The HR team has received multiple candidate resumes.


Manually comparing every resume with the job description takes time and may lead to inconsistency.


To make the process structured, we will create a Resume Fitment Scoring System that will:


a) compare resume text with the job description calculate similarity score

b) include structured features such as skills, experience, education, and certifications

c)compute a final fitment score

d)rank candidates from best to least suitable

In [1]:
# ============================================================
# CELL 2: IMPORT REQUIRED LIBRARIES
# ============================================================
# In this cell, we import all the Python libraries
# needed for data handling and text similarity.
# ============================================================

import pandas as pd
import numpy as np

# TfidfVectorizer converts text into numerical form
from sklearn.feature_extraction.text import TfidfVectorizer

# cosine_similarity helps compare resume text with job description
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# ============================================================
# CELL 4: CREATE SAMPLE CANDIDATE DATASET
# ============================================================
# We are creating a small dataset directly in Python.
# Later, this can also be replaced with a CSV file upload.
# ============================================================

data = {
    "candidate_id": [1, 2, 3, 4, 5],
    "name": ["Aman", "Bhavna", "Charu", "Deepak", "Esha"],
    "skills": [
        "Python SQL Excel Power BI",
        "Excel HRMS Recruitment Communication",
        "Python SQL Machine Learning Tableau",
        "SQL Excel Data Cleaning Reporting",
        "Python SQL Excel Tableau Statistics"
    ],
    "experience_years": [3, 5, 2, 4, 3],
    "education": [
        "B.Tech Computer Science",
        "MBA HR",
        "B.Sc Data Science",
        "B.Com",
        "M.Sc Statistics"
    ],
    "certifications": [
        "Google Data Analytics",
        "HR Analytics Certificate",
        "Machine Learning Certificate",
        "Excel Advanced",
        "Power BI and Statistics"
    ],
    "resume_text": [
        "Experienced in Python SQL Excel and Power BI. Built dashboards and cleaned HR data.",
        "Worked in recruitment onboarding HR operations and employee engagement using Excel and HRMS.",
        "Strong in Python SQL machine learning and Tableau. Worked on predictive analytics projects.",
        "Experienced in SQL Excel reporting and data cleaning. Prepared weekly business reports.",
        "Skilled in Python SQL Excel Tableau and statistics. Built reports dashboards and data summaries."
    ]
}

# Convert dictionary to pandas DataFrame
df = pd.DataFrame(data)

# Display the dataset
df

,candidate_id,name,skills,experience_years,education,certifications,resume_text
0,1,Aman,Python SQL Excel Power BI,3,B.Tech Computer Science,Google Data Analytics,Experienced in Python SQL Excel and Power BI. ...
1,2,Bhavna,Excel HRMS Recruitment Communication,5,MBA HR,HR Analytics Certificate,Worked in recruitment onboarding HR operations...
2,3,Charu,Python SQL Machine Learning Tableau,2,B.Sc Data Science,Machine Learning Certificate,Strong in Python SQL machine learning and Tabl...
3,4,Deepak,SQL Excel Data Cleaning Reporting,4,B.Com,Excel Advanced,Experienced in SQL Excel reporting and data cl...
4,5,Esha,Python SQL Excel Tableau Statistics,3,M.Sc Statistics,Power BI and Statistics,Skilled in Python SQL Excel Tableau and statis...


In [3]:
# ============================================================
# CELL 6: DEFINE THE JOB DESCRIPTION
# ============================================================
# This job description represents the role the company
# wants to fill. Candidate resumes will be compared against it.
# ============================================================

job_description = """
We are hiring a Data Analyst with skills in Python, SQL, Excel, Power BI,
data cleaning, dashboard creation, reporting, and statistical analysis.
The candidate should be able to analyze business data, create visual reports,
and support data-driven decision making.
"""

print(job_description)


We are hiring a Data Analyst with skills in Python, SQL, Excel, Power BI,
data cleaning, dashboard creation, reporting, and statistical analysis.
The candidate should be able to analyze business data, create visual reports,
and support data-driven decision making.



In [4]:
# ============================================================
# CELL 8: CALCULATE TEXT SIMILARITY
# ============================================================
# Here we compare each resume with the job description.
#
# Step 1: Put job description and all resume texts together
# Step 2: Convert text to TF-IDF vectors
# Step 3: Compute cosine similarity
# ============================================================

# Create a list containing the job description and all resumes
all_texts = [job_description] + df["resume_text"].tolist()

# Create TF-IDF vectorizer object
vectorizer = TfidfVectorizer()

# Convert all texts into TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(all_texts)

# The first row is the job description
# Remaining rows are the resumes
job_vector = tfidf_matrix[0:1]
resume_vectors = tfidf_matrix[1:]

# Calculate cosine similarity between job description and each resume
similarity_scores = cosine_similarity(job_vector, resume_vectors)

# Flatten the result and store it in the dataframe
df["text_similarity_score"] = similarity_scores.flatten()

# Display candidate names with similarity scores
df[["name", "text_similarity_score"]]

,name,text_similarity_score
0,Aman,0.292522
1,Bhavna,0.068751
2,Charu,0.067273
3,Deepak,0.337626
4,Esha,0.230427


In [5]:
# ============================================================
# CELL 10: CREATE FEATURE-BASED SCORES
# ============================================================
# We now assign scores to different candidate attributes.
# These are simplified classroom rules for understanding.
# ============================================================

# ------------------------------------------------------------
# SKILLS SCORE
# ------------------------------------------------------------
# We will check whether important keywords are present
# in the candidate's skills field.

required_skills = ["Python", "SQL", "Excel", "Power BI", "Statistics"]

def calculate_skills_score(skill_text):
    """
    This function checks how many required skills
    are present in the candidate's skill list.
    """
    score = 0

    for skill in required_skills:
        if skill.lower() in skill_text.lower():
            score += 1

    # Convert to a score out of 10
    return (score / len(required_skills)) * 10


# ------------------------------------------------------------
# EXPERIENCE SCORE
# ------------------------------------------------------------
# We define a simple rule:
# 3 or more years = stronger fit for this role

def calculate_experience_score(years):
    """
    This function gives a score out of 10
    based on years of experience.
    """
    if years >= 5:
        return 10
    elif years >= 3:
        return 8
    elif years >= 2:
        return 6
    else:
        return 4


# ------------------------------------------------------------
# EDUCATION SCORE
# ------------------------------------------------------------
# Relevant education gets higher score.

def calculate_education_score(education_text):
    """
    This function scores education relevance out of 10.
    """
    education_text = education_text.lower()

    if "data" in education_text or "statistics" in education_text or "computer" in education_text:
        return 10
    elif "mba" in education_text:
        return 7
    else:
        return 5


# ------------------------------------------------------------
# CERTIFICATION SCORE
# ------------------------------------------------------------
# Certifications relevant to data analytics get higher score.

def calculate_certification_score(cert_text):
    """
    This function scores certifications out of 10.
    """
    cert_text = cert_text.lower()

    if "data" in cert_text or "power bi" in cert_text or "machine learning" in cert_text:
        return 10
    elif "excel" in cert_text:
        return 7
    else:
        return 5


# Apply the scoring functions
df["skills_score"] = df["skills"].apply(calculate_skills_score)
df["experience_score"] = df["experience_years"].apply(calculate_experience_score)
df["education_score"] = df["education"].apply(calculate_education_score)
df["certification_score"] = df["certifications"].apply(calculate_certification_score)

# Display the scores
df[[
    "name",
    "skills_score",
    "experience_score",
    "education_score",
    "certification_score"
]]

,name,skills_score,experience_score,education_score,certification_score
0,Aman,8.0,8,10,10
1,Bhavna,2.0,10,7,5
2,Charu,4.0,6,10,10
3,Deepak,4.0,8,5,7
4,Esha,8.0,8,10,10


In [6]:
# ============================================================
# CELL 12: DEFINE WEIGHTS FOR FINAL FITMENT SCORE
# ============================================================
# These weights represent the importance of each factor.
# Total should add up to 1.0
# ============================================================

weight_text_similarity = 0.30
weight_skills = 0.30
weight_experience = 0.20
weight_education = 0.10
weight_certification = 0.10

In [7]:
# ============================================================
# CELL 14: SCALE TEXT SIMILARITY TO 0-10
# ============================================================
# Since cosine similarity gives values from 0 to 1,
# we multiply by 10 so that it matches the same scale
# as the other feature scores.
# ============================================================

df["text_similarity_score_10"] = df["text_similarity_score"] * 10

# Display scaled similarity score
df[["name", "text_similarity_score", "text_similarity_score_10"]]

,name,text_similarity_score,text_similarity_score_10
0,Aman,0.292522,2.925221
1,Bhavna,0.068751,0.687511
2,Charu,0.067273,0.672732
3,Deepak,0.337626,3.376258
4,Esha,0.230427,2.304267


In [8]:
# ============================================================
# CELL 16: CALCULATE FINAL FITMENT SCORE
# ============================================================
# Final Score =
# (Text Similarity × weight) +
# (Skills Score × weight) +
# (Experience Score × weight) +
# (Education Score × weight) +
# (Certification Score × weight)
# ============================================================

df["final_fitment_score"] = (
    df["text_similarity_score_10"] * weight_text_similarity +
    df["skills_score"] * weight_skills +
    df["experience_score"] * weight_experience +
    df["education_score"] * weight_education +
    df["certification_score"] * weight_certification
)

# Display final score
df[["name", "final_fitment_score"]]

,name,final_fitment_score
0,Aman,6.877566
1,Bhavna,4.006253
2,Charu,4.601820
3,Deepak,5.012877
4,Esha,6.691280


In [9]:
# ============================================================
# CELL 18: RANK CANDIDATES
# ============================================================
# We sort candidates in descending order
# so that the highest scoring candidate appears first.
# ============================================================

ranked_df = df.sort_values(by="final_fitment_score", ascending=False).reset_index(drop=True)

# Add rank column starting from 1
ranked_df["rank"] = ranked_df.index + 1

# Display the final ranking
ranked_df[[
    "rank",
    "name",
    "text_similarity_score_10",
    "skills_score",
    "experience_score",
    "education_score",
    "certification_score",
    "final_fitment_score"
]]

,rank,name,text_similarity_score_10,skills_score,experience_score,education_score,certification_score,final_fitment_score
0,1,Aman,2.925221,8.0,8,10,10,6.877566
1,2,Esha,2.304267,8.0,8,10,10,6.691280
2,3,Deepak,3.376258,4.0,8,5,7,5.012877
3,4,Charu,0.672732,4.0,6,10,10,4.601820
4,5,Bhavna,0.687511,2.0,10,7,5,4.006253


In [10]:
# ============================================================
# CELL 20: SHORTLIST TOP 3 CANDIDATES
# ============================================================
# This cell selects the top 3 candidates
# based on final fitment score.
# ============================================================

top_3_candidates = ranked_df.head(3)

top_3_candidates[["rank", "name", "final_fitment_score"]]

,rank,name,final_fitment_score
0,1,Aman,6.877566
1,2,Esha,6.691280
2,3,Deepak,5.012877


In [11]:
# ============================================================
# CELL 22: DISPLAY COMPLETE FINAL RESULT TABLE
# ============================================================
# This table gives a complete view of all scores
# for all candidates.
# ============================================================

final_result = ranked_df[[
    "rank",
    "candidate_id",
    "name",
    "text_similarity_score_10",
    "skills_score",
    "experience_score",
    "education_score",
    "certification_score",
    "final_fitment_score"
]]

final_result

,rank,candidate_id,name,text_similarity_score_10,skills_score,experience_score,education_score,certification_score,final_fitment_score
0,1,1,Aman,2.925221,8.0,8,10,10,6.877566
1,2,5,Esha,2.304267,8.0,8,10,10,6.691280
2,3,4,Deepak,3.376258,4.0,8,5,7,5.012877
3,4,3,Charu,0.672732,4.0,6,10,10,4.601820
4,5,2,Bhavna,0.687511,2.0,10,7,5,4.006253


In [12]:
# ============================================================
# CELL 24: SAVE FINAL RESULT AS CSV
# ============================================================
# This cell saves the final ranking into a CSV file
# so students can download and review the result.
# ============================================================

# Save file
final_result.to_csv("resume_fitment_scoring_result.csv", index=False)

print("File saved successfully")

# Download file
from google.colab import files
files.download("resume_fitment_scoring_result.csv")

File saved successfully


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>